# Laboratorio 2 · Auditoría de un pipeline de datos

**Explorar → Analizar → Experimentar → Decidir → Justificar → Reflexionar**

Leyes, Ética y Protección de Datos · Especialización en Análisis Estadístico para Ciencia de Datos · Docente: Wilson Sandoval Rodríguez

---

> **Versión del estudiante.** Las celdas están sin ejecutar y hay bloques marcados con `TODO` que usted debe completar. Ejecute de arriba hacia abajo y responda cada pregunta antes de continuar.

> Todos los datos son **sintéticos**. Ninguna persona real está representada.


## Qué va a hacer aquí

| | |
|:--|:--|
| **Aprenderá a** | Auditar un pipeline etapa por etapa y comparar dos modelos que difieren en desempeño y en lo que están dispuestos a usar |
| **Duración** | 40 minutos |
| **Evidencia** | La matriz de auditoría, el registro de decisión y sus respuestas |
| **Unidad** | 1 · Fundamentos legales (prepara la Unidad 2) |

**No es un ejercicio de rendimiento predictivo.** Las métricas están aquí para
sostener una discusión, no para ganar una competencia.

---

# 1 · EXPLORAR

*¿De dónde vienen estos datos?*

In [ ]:
from pathlib import Path
import pandas as pd

# Funciona en tres sitios sin cambiar nada:
#   - dentro del repositorio (labs/ o raíz)
#   - en Google Colab
#   - en cualquier equipo con internet
RUTA = Path("../data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = Path("data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = "https://raw.githubusercontent.com/wilsonsr/leyes-etica-proteccion-datos/main/data/clientes_sinteticos.csv"

print("Origen de los datos:", RUTA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

df = pd.read_csv(RUTA, parse_dates=["fecha_ultima_compra", "fecha_autorizacion"])
print(f"{df.shape[0]} registros · {df.shape[1]} variables")

In [ ]:
origen = (
    df["origen_dato"].value_counts().to_frame("registros")
      .assign(**{"%": lambda d: (d["registros"] / len(df) * 100).round(1)})
)
origen["con_autorizacion_fechada"] = df.groupby("origen_dato")["fecha_autorizacion"].apply(lambda s: s.notna().sum())
origen["consent_marketing"] = df.groupby("origen_dato")["consentimiento_marketing"].sum()
origen

::: {.rds-card .riesgo}
**Pregunta 1.** Dos orígenes no tienen ni una sola autorización fechada. ¿Cuáles son y cuántos registros suman?

Fíjese también en `consent_marketing`: la autorización de marketing no se distribuye de forma pareja. ¿Qué implicación tiene para la petición de «publicidad personalizada» que llegó esta semana?
:::

In [ ]:
UE = {"Espana", "Alemania", "Francia", "Italia"}
territorio = df["pais_residencia"].value_counts().to_frame("registros")
territorio["régimen probable"] = np.where(
    territorio.index.isin(UE), "GDPR (UE)",
    np.where(territorio.index == "Colombia", "Ley 1581 de 2012", "otro · verificar"))
territorio

::: {.rds-card .decide}
**Pregunta 2.** ¿Cuántos residentes en la Unión Europea hay? Es un número pequeño frente al total.

¿Cambia algo que sean pocos? ¿Por qué sí o por qué no?
:::

---

# 2 · ANALIZAR

*¿Qué hace el pipeline con ellos?*

### Aislar, no borrar

Los registros sin evidencia de origen se marcan y se separan. Borrarlos
destruiría la prueba de que existieron.

In [ ]:
SIN_EVIDENCIA = ["lista_comprada_tercero", "enriquecimiento_web"]
df["evidencia_origen"] = np.where(
    df["origen_dato"].isin(SIN_EVIDENCIA), "sin evidencia", "con evidencia")

df_apto = df[df["evidencia_origen"] == "con evidencia"].copy()
df_aislado = df[df["evidencia_origen"] == "sin evidencia"].copy()

print(f"Base original            : {len(df):>5}")
print(f"Aptos para entrenamiento : {len(df_apto):>5}")
print(f"Aislados                 : {len(df_aislado):>5}  ({len(df_aislado)/len(df)*100:.1f} %)")

::: {.rds-card .decide}
**Pregunta 3.** Perdimos un 15 % de los registros. La Dirección Comercial va a preguntar por qué el modelo se entrenó con menos datos.

Escriba la respuesta en **dos frases**, sin usar la palabra «ley».
:::

### Feature engineering: las inferencias que creamos nosotros

Esta es la etapa donde el analista **fabrica** datos nuevos sobre personas.

In [ ]:
CORTE = pd.Timestamp("2026-08-31")   # fecha de corte del análisis

df_apto["dias_desde_ultima_compra"] = (CORTE - df_apto["fecha_ultima_compra"]).dt.days
df_apto["ticket_promedio"] = np.where(
    df_apto["compras_6m"] > 0, df_apto["monto_compras"] / df_apto["compras_6m"], 0)
df_apto["intensidad_navegacion"] = (
    df_apto["productos_vistos"] / df_apto["visitas_web"].clip(lower=1))

df_apto[["dias_desde_ultima_compra", "ticket_promedio", "intensidad_navegacion"]].describe().round(2)

::: {.rds-card .decide}
**Pregunta 4.** Las tres variables anteriores son inofensivas.

Ahora piense en tres más que el equipo de marketing *podría* pedir y que no lo serían. Escríbalas con el nombre que tendrían en el código.

Pista: con `compras_farmacia_6m`, `busquedas_maternidad` y `edad` se pueden construir varias.
:::

### Una advertencia estadística antes de seguir

En el laboratorio anterior miramos correlaciones. Conviene ser preciso:

::: {.callout-warning}
## Tres errores de lectura que hay que evitar

**Correlación baja no significa variable inútil.** Una variable con correlación
lineal cercana a cero puede ser muy predictiva en interacción con otras, o de
forma no lineal. Descartar variables por su correlación marginal es una mala
práctica de modelado, además de una mala justificación de minimización: la
minimización se argumenta por **finalidad**, no por correlación.

**Correlación alta no significa causa.** Que `dias_desde_ultima_compra`
prediga el abandono no significa que esperar cause abandono. El modelo describe
asociación en estos datos, no un mecanismo.

**Cuidado con la fuga de información.** `score_riesgo_interno` y `segmento` se
derivan de un modelo anterior de la propia empresa. Usarlos como predictores
mezcla la salida de un sistema con la entrada de otro, infla artificialmente el
desempeño y hace imposible auditar de dónde salió una decisión. Por eso quedan
**fuera de los dos modelos** de este laboratorio.
:::

---

# 3 · EXPERIMENTAR

*Dos modelos, una diferencia incómoda*

In [ ]:
VARIABLES_A = [
    "visitas_web", "productos_vistos", "compras_6m", "monto_compras",
    "dias_desde_ultima_compra", "ticket_promedio", "intensidad_navegacion",
]

VARIABLES_EXTRA_B = [
    "edad",                   # cuasi-identificador
    "estrato",                # proxy socioeconómico
    "ingresos_mensuales",     # proxy socioeconómico
    "compras_farmacia_6m",    # proxy de salud
    "entrega_asistida",       # proxy de discapacidad o edad avanzada
    "busquedas_maternidad",   # proxy de embarazo
]

VARIABLES_B = VARIABLES_A + VARIABLES_EXTRA_B
print(f"Modelo A: {len(VARIABLES_A)} variables (solo comportamiento)")
print(f"Modelo B: {len(VARIABLES_B)} variables (+ demografía y proxies)")

In [ ]:
y = df_apto["churn"]


def entrenar(variables, etiqueta, semilla=42):
    X = df_apto[variables].copy()
    X = X.fillna(X.median(numeric_only=True))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, random_state=semilla, stratify=y)
    modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    modelo.fit(X_tr, y_tr)
    p_te = modelo.predict_proba(X_te)[:, 1]
    return {"etiqueta": etiqueta, "modelo": modelo, "variables": variables,
            "auc": roc_auc_score(y_te, p_te), "X_te": X_te, "y_te": y_te, "p_te": p_te}


modelo_a = entrenar(VARIABLES_A, "Modelo A · solo comportamiento")
modelo_b = entrenar(VARIABLES_B, "Modelo B · + demografía y proxies")

print(f"{modelo_a['etiqueta']:<42} AUC = {modelo_a['auc']:.3f}")
print(f"{modelo_b['etiqueta']:<42} AUC = {modelo_b['auc']:.3f}")
print(f"{'Diferencia':<42}     + {modelo_b['auc'] - modelo_a['auc']:.3f}")

In [ ]:
fig, ax = plt.subplots()
for m, color in [(modelo_a, "#0e7490"), (modelo_b, "#db2777")]:
    fpr, tpr, _ = roc_curve(m["y_te"], m["p_te"])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{m['etiqueta']}  (AUC {m['auc']:.3f})")
ax.plot([0, 1], [0, 1], color="#94a3b8", lw=1, ls="--", label="Azar")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curvas ROC · modelo A vs modelo B", loc="left", fontsize=11)
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

### ¿Qué variables aportan la diferencia?

In [ ]:
coefs = pd.Series(
    modelo_b["modelo"].named_steps["logisticregression"].coef_[0],
    index=VARIABLES_B, name="coeficiente").sort_values(key=abs, ascending=False)

tabla = coefs.to_frame().round(3)
tabla["tipo"] = np.where(tabla.index.isin(VARIABLES_EXTRA_B),
                         "demografía / proxy", "comportamiento")
tabla

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5))
colores = ["#db2777" if v in VARIABLES_EXTRA_B else "#0e7490" for v in coefs.index]
ax.barh(coefs.index[::-1], coefs.values[::-1], color=colores[::-1])
ax.axvline(0, color="#475569", lw=1)
ax.set_title("Modelo B · peso de cada variable (coeficientes estandarizados)",
             loc="left", fontsize=11)
ax.set_xlabel("← menos probabilidad de abandono    |    más probabilidad de abandono →")
plt.tight_layout(); plt.show()

::: {.rds-card .riesgo}
**Pregunta 5.** Mire las barras magenta: son las variables que el modelo A no tiene.

Si alguna está entre las de mayor peso, el modelo B no está prediciendo mejor *el comportamiento*: está prediciendo mejor *la condición social o de salud de la persona*, y usándola para decidir sobre ella.

¿Cuál es la variable de mayor peso del bloque magenta? ¿Se siente cómodo explicándosela a un cliente?
:::

### ¿Sobre quién se equivoca cada modelo? {#por-grupo}

In [ ]:
UMBRAL = 0.50


def error_por_grupo(m, variable_grupo):
    grupo = df_apto.loc[m["X_te"].index, variable_grupo]
    pred = (m["p_te"] >= UMBRAL).astype(int)
    ev = pd.DataFrame({"grupo": grupo, "real": m["y_te"].values, "pred": pred})
    filas = []
    for nombre, sub in ev.groupby("grupo", dropna=True):
        if len(sub) < 20:       # grupos muy pequeños: no se reporta
            continue
        tn, fp, fn, tp = confusion_matrix(sub["real"], sub["pred"], labels=[0, 1]).ravel()
        filas.append({"grupo": nombre, "n": len(sub),
                      "tasa_churn_real": round(sub["real"].mean(), 3),
                      "marcados_como_riesgo": round(sub["pred"].mean(), 3),
                      "falsos_positivos": round(fp / max(tn + fp, 1), 3),
                      "falsos_negativos": round(fn / max(tp + fn, 1), 3)})
    return pd.DataFrame(filas).set_index("grupo")


print("=== MODELO A · por estrato ===")
print(error_por_grupo(modelo_a, "estrato").to_string())
print()
print("=== MODELO B · por estrato ===")
print(error_por_grupo(modelo_b, "estrato").to_string())

::: {.rds-card .riesgo}
**Pregunta 6.** Compare la columna `marcados_como_riesgo` entre estratos en cada modelo.

Si el modelo B marca como «en riesgo» a una proporción mucho mayor de clientes de estratos bajos, y la acción asociada es *dar menos beneficios a quien se va a ir de todos modos*, el modelo acaba de convertirse en un mecanismo de exclusión con apariencia técnica.

¿Ocurre aquí? Responda con los números de la tabla, no con la intuición.
:::

---

# 4 · DECIDIR {#decidir}

*La decisión no está en el modelo*

In [ ]:
acciones = pd.DataFrame([
    dict(accion="Oferta de retención (descuento)",
         falso_positivo="Descuento a quien no se iba a ir → costo para la empresa",
         falso_negativo="Se pierde un cliente retenible → costo para la empresa",
         dano="La empresa"),
    dict(accion="Menor prioridad en atención al cliente",
         falso_positivo="Peor servicio a quien no se iba a ir → daño a la persona",
         falso_negativo="Servicio normal a quien se va → costo menor",
         dano="La persona"),
    dict(accion="Precio personalizado más alto",
         falso_positivo="Sobreprecio a quien no se iba a ir → daño a la persona",
         falso_negativo="Precio normal → sin daño",
         dano="La persona"),
])
for _, f in acciones.iterrows():
    print(f"\n▸ {f['accion']}")
    print(f"   FP: {f['falso_positivo']}")
    print(f"   FN: {f['falso_negativo']}")
    print(f"   Daño: lo asume {f['dano'].lower()}")

::: {.rds-card .dilema}
**Pregunta 7.** El modelo es exactamente el mismo en las tres filas. Lo que cambia es quién asume el daño del error.

Escriba la decisión final:

1. ¿Qué modelo se despliega, A o B?
2. ¿Con qué acción asociada?
3. ¿Qué condición tendría que cumplirse para cambiar de opinión?
:::

---

# 5 · JUSTIFICAR

*La auditoría como artefacto del proyecto*

La matriz de auditoría no es un documento de Word que nadie actualiza: se
construye en código y se versiona con el proyecto.

In [ ]:
# TODO: complete las filas que faltan con SU análisis.

auditoria = pd.DataFrame([
    dict(etapa="Origen", dato="origen_dato, fecha_autorizacion",
         riesgo="236 registros sin evidencia de autorización",
         pregunta="¿De dónde vienen y para qué se recogieron?",
         control="Aislar los registros sin evidencia",
         decision="Excluidos del entrenamiento; conservados aparte"),
    dict(etapa="Datos", dato="identificadores directos",
         riesgo="TODO", pregunta="TODO", control="TODO", decision="TODO"),
    dict(etapa="Modelo", dato="estrato, ingresos, proxies de salud",
         riesgo="TODO", pregunta="TODO", control="TODO", decision="TODO"),
    dict(etapa="Decisión", dato="acción comercial asociada al score",
         riesgo="TODO", pregunta="TODO", control="TODO", decision="TODO"),
    dict(etapa="Publicación", dato="tablero y archivo a terceros",
         riesgo="TODO", pregunta="TODO", control="TODO", decision="TODO"),
])
auditoria

::: {.rds-card .decision}
**Pregunta 8.** De las ocho etapas, ¿cuál es la que en su organización real está peor documentada?

No la que tiene más riesgo: la que nadie escribió nunca.
:::

---

# 6 · REFLEXIONAR

*¿Y en mi trabajo?*

::: {.rds-card .reflexiona}
**Pregunta 9 · de salida.**

> **¿Qué tendría que cambiar en mi proyecto de datos?**

Hoy con una restricción: la respuesta debe nombrar **una etapa del pipeline** y **un cambio ejecutable**. No vale «ser más cuidadoso».
:::

---

## Lo que queda del laboratorio

- `auditoria_pipeline_datamarket.csv` — la matriz de las ocho etapas.
- Su decisión sobre qué modelo desplegar, con qué acción y bajo qué condición.

Ninguno de los dos es un documento legal. Los dos son **evidencia de
ingeniería**: si en seis meses alguien pregunta por qué el modelo no usa
estrato, la respuesta existe y tiene fecha.

**Marco legal relacionado:** decisiones automatizadas y perfilamiento — GDPR
art. 22, LGPD art. 20, y el proyecto de reforma colombiano.

**Siguiente paso:** Actividad 1 (tres países) y Actividad 2 (GDPR).